# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [74]:
event_log_name = "medium"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")

Event log name medium
LTN Rows path medium_ltn_rows.pkl


In [75]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


In [76]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.mediumevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(medium_leaky_row_classes)

[<class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-10'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-25'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-50'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-100'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-150'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-200'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-250'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-300'>, <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-350'>]


In [77]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)


Created directory: d:\LTNcoder\.out\plots\medium_evaluations_both_2025-08-07-11-42-07
Deleted all rows from Evaluation and Model tables.


# Training

In [78]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [79]:
ads = [
        dict(ad=MediumDAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in medium_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in medium_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-200'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-250'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april

Fitting ADs:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1/6
42/42 [==============================] - 1s 13ms/step - loss: 0.2154 - accuracy: 0.1569 - val_loss: 0.1213 - val_accuracy: 1.0000
Epoch 2/6
42/42 [==============================] - 0s 7ms/step - loss: 0.0285 - accuracy: 0.6016 - val_loss: 0.0050 - val_accuracy: 1.0000
Epoch 3/6
42/42 [==============================] - 0s 7ms/step - loss: 0.0051 - accuracy: 0.5937 - val_loss: 0.0048 - val_accuracy: 1.0000
Epoch 4/6
42/42 [==============================] - 0s 7ms/step - loss: 0.0050 - accuracy: 0.5951 - val_loss: 0.0047 - val_accuracy: 1.0000
Epoch 5/6
42/42 [==============================] - 0s 8ms/step - loss: 0.0049 - accuracy: 0.5949 - val_loss: 0.0047 - val_accuracy: 1.0000
Epoch 6/6
42/42 [==============================] - 0s 7ms/step - loss: 0.0048 - accuracy: 0.5819 - val_loss: 0.0046 - val_accuracy: 1.0000
d:\LTNcoder\.out\models\medium-0.3-1_mediumdae_20250807-114207.319330.keras
Loading model medium-0.3-1_mediumdae_20250807-114207.319330 / <april.fs.ModelFile object 

In [80]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'likelihood+': <class 'april.anomalydetection.boehmer.LikelihoodPlusAnomalyDetector'>, 'mediumdae': <class 'april.anomalydetection.mediumencoder.MediumDAE'>, 'mediumdae-leaky-10': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-10'>, 'mediumdae-leaky-100': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-100'>, 'mediumdae-leaky-150': <class 'april.anomalydetection.mediumencoder.MediumDAE-Leaky-150'>, 'mediumdae-leaky-200': <cl

# Evaluation

In [81]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [82]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [83]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697', 'medium-0.3-1_mediumdae-leaky-10_20250807-114210.593361', 'medium-0.3-1_mediumdae-leaky-150_20250807-114222.687737', 'medium-0.3-1_mediumdae-leaky-200_20250807-114226.084445', 'medium-0.3-1_mediumdae-leaky-250_20250807-114229.144818', 'medium-0.3-1_mediumdae-leaky-25_20250807-114213.715398', 'medium-0.3-1_mediumdae-leaky-300_20250807-114232.117905', 'medium-0.3-1_mediumdae-leaky-350_20250807-114235.288186', 'medium-0.3-1_mediumdae-leaky-50_20250807-114216.743181', 'medium-0.3-1_mediumdae_20250807-114207.319330', 'medium-0.3-1_mediumltnfrozen-100_20250807-114329.272294', 'medium-0.3-1_mediumltnfrozen-10_20250807-114238.553896', 'medium-0.3-1_mediumltnfrozen-150_20250807-114346.277587', 'medium-0.3-1_mediumltnfrozen-200_20250807-114412.565492', 'medium-0.3-1_mediumltnfrozen-250_20250807-114439.375275', 'medium-0.3-1_mediumltnfrozen-25_20250807-114255.545031', 'medium-0.3-1_mediumltnfrozen-300_20250807-114504.45

Evaluate:   0%|          | 0/19 [00:00<?, ?it/s]

Evaluating medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697...
Loading model medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697 / <april.fs.ModelFile object at 0x000001A9A684A6D0> for event log medium-0.3-1 at path d:\LTNcoder\.out\models\medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697.keras
Self.ad_: <april.anomalydetection.mediumencoder.MediumDAE-Leaky-100 object at 0x000001A9A684A730>
<april.mediumevaluator.Evaluator object at 0x000001A9A684A100> loaded.
e.model_file: d:\LTNcoder\.out\models\medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697.keras
e.model_name: medium-0.3-1_mediumdae-leaky-100_20250807-114219.715697
e.eventlog_name: medium-0.3-1
Filtering dataset to 375 LTN rows.
Indices: [10, 17, 21, 41, 59, 61, 85, 92, 95, 97, 127, 139, 148, 150, 164, 189, 195, 204, 220, 237, 254, 271, 289, 292, 293, 299, 307, 309, 323, 330, 358, 374, 376, 392, 394, 398, 409, 412, 413, 418, 493, 497, 498, 511, 514, 515, 537, 578, 585, 605, 654, 658, 665, 690, 696, 713, 725

In [84]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/5472 [00:00<?, ?it/s]

In [85]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['MediumDAE', 'MediumDAE-Leaky-10', 'MediumDAE-Leaky-25', 'MediumDAE-Leaky-50', 'MediumDAE-Leaky-100', 'MediumDAE-Leaky-150', 'MediumDAE-Leaky-200', 'MediumDAE-Leaky-250', 'MediumDAE-Leaky-300', 'MediumDAE-Leaky-350', 'MediumLTNFROZEN-10', 'MediumLTNFROZEN-25', 'MediumLTNFROZEN-50', 'MediumLTNFROZEN-100', 'MediumLTNFROZEN-150', 'MediumLTNFROZEN-200', 'MediumLTNFROZEN-250', 'MediumLTNFROZEN-300', 'MediumLTNFROZEN-350']


In [86]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [87]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [88]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [89]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [90]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

    axis                   ad process_model  dataset_name        f1  \
0   Case            MediumDAE        Medium  medium-0.3-1  0.225352   
1   Case   MediumDAE-Leaky-10        Medium  medium-0.3-1  0.249848   
2   Case  MediumDAE-Leaky-100        Medium  medium-0.3-1  0.225352   
3   Case  MediumDAE-Leaky-150        Medium  medium-0.3-1  0.225352   
4   Case  MediumDAE-Leaky-200        Medium  medium-0.3-1  0.225352   
5   Case   MediumDAE-Leaky-25        Medium  medium-0.3-1  0.225352   
6   Case  MediumDAE-Leaky-250        Medium  medium-0.3-1  0.270354   
7   Case  MediumDAE-Leaky-300        Medium  medium-0.3-1  0.225352   
8   Case  MediumDAE-Leaky-350        Medium  medium-0.3-1  0.369654   
9   Case   MediumDAE-Leaky-50        Medium  medium-0.3-1  0.225352   
10  Case   MediumLTNFROZEN-10        Medium  medium-0.3-1  0.441131   
11  Case  MediumLTNFROZEN-100        Medium  medium-0.3-1  0.424676   
12  Case  MediumLTNFROZEN-150        Medium  medium-0.3-1  0.463562   
13  Ca

C:\Users\devas\AppData\Local\Temp\ipykernel_69200\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_69200\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


In [91]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,MediumDAE,Medium,medium-0.3-1,0.225352,0.500000,0.145455
1,Case,MediumDAE-Leaky-10,Medium,medium-0.3-1,0.249848,0.583333,0.158968
2,Case,MediumDAE-Leaky-100,Medium,medium-0.3-1,0.225352,0.500000,0.145455
3,Case,MediumDAE-Leaky-150,Medium,medium-0.3-1,0.225352,0.500000,0.145455
4,Case,MediumDAE-Leaky-200,Medium,medium-0.3-1,0.225352,0.500000,0.145455
5,Case,MediumDAE-Leaky-25,Medium,medium-0.3-1,0.225352,0.500000,0.145455
6,Case,MediumDAE-Leaky-250,Medium,medium-0.3-1,0.270354,0.625000,0.172482
7,Case,MediumDAE-Leaky-300,Medium,medium-0.3-1,0.225352,0.500000,0.145455
8,Case,MediumDAE-Leaky-350,Medium,medium-0.3-1,0.369654,0.681818,0.253563
9,Case,MediumDAE-Leaky-50,Medium,medium-0.3-1,0.225352,0.500000,0.145455
